# Lab 3. Surface code — Reading error signatures and protecting logical information

## Introduction

In this lab, you will <strong>measure signs of errors without directly reading the stored quantum information, then choose a correction from those measurement results</strong>. After applying the correction, you will separately check whether the stabilizer outcomes have returned to their expected values and whether the original information has been restored.

The three-qubit repetition code in the review material stored $\alpha|0\rangle+\beta|1\rangle$ as $\alpha|000\rangle+\beta|111\rangle$. It encoded one piece of quantum information in a joint state of several physical qubits; it did not make several copies of the same state. Quantum information stored in this way forms a <strong>logical qubit</strong>. Writing its two logical basis states as $|0_L\rangle,|1_L\rangle$, the state to protect is

$$
|\psi_L\rangle=\alpha|0_L\rangle+\beta|1_L\rangle
$$

The subscript L denotes a logical state.

Measuring the data qubits individually can distinguish the logical components as well as reveal errors, destroying the superposition. Instead, we measure relations shared by all valid logical states. The corresponding check operator is called a <strong>stabilizer</strong>, and an ordered record of several check outcomes is called a <strong>syndrome</strong>. In Problem 1, a small parity-measurement circuit will show why such a measurement can preserve the stored information.

We will use a small <strong>surface code</strong> that stores one logical qubit in nine data qubits. Four X-type and four Z-type checks allow us to address X, Y, and Z errors. Tables specify the data-qubit numbers and the support of each check. The provided tools prepare logical states and calculate ideal measurements on the full patch.

The classical rule that selects a correction from a syndrome is called a <strong>decoder</strong>. This is different from undoing the encoding circuit. The data remains encoded, and the decoder chooses a correction from the syndrome alone, without receiving the actual error or the target state. Information about the original state is used only to verify the result after correction.

| Problem | Your task |
|---|---|
| <strong>1. Measuring while preserving information</strong> | Check both the ancilla outcome and the data state after measurement. |
| <strong>2. Errors and syndromes</strong> | Use the relation between an error and a check to determine which check signs change. |
| <strong>3. Choosing a correction from a syndrome</strong> | Examine the allowed error candidates and explain why different corrections can be equivalent. |
| <strong>4. Verifying logical recovery</strong> | Distinguish a zero syndrome from restoration of the original logical state. |

In the basic experiments, a valid code state experiences <strong>an X, Y, or Z error on at most one data qubit</strong>. State preparation, checks, and correction are ideal. Problem 4 and the final application introduce errors on two data qubits so that you can examine how to interpret results outside this assumption.

As in the joint-state verification in Lab 1 and the performance comparisons in Lab 2, distinguish what you observe from what those observations justify. The goal is <strong>to restore the originally stored logical information</strong>, rather than merely to identify the error by name.

## How to use this notebook

Each section presents <strong>the purpose of the calculation and definitions of the notation, key equations, worked examples, and exercises</strong>. Before reading an equation, identify the states and operations involved. In particular, distinguish the data qubits that store information, the ancilla qubits used to read check outcomes, and the classical bits that record the syndrome.

Run the code cells in order from top to bottom. Later cells use functions and variables defined earlier, so starting in the middle may cause errors. Also run cells marked as provided code. They contain tools for state preparation, error application, measurement calculations, and table output. Each tool is introduced where it is needed; you do not need to analyze every implementation detail before starting the exercises.

For each problem, follow these steps.

1. <strong>Read the problem and predict the result.</strong> Identify what information is measured, the assumed error model, and the information available when choosing a correction. When a question asks for a prediction, write your prediction and reasoning before running the experiment.
2. <strong>Complete the TODO.</strong> Replace `raise NotImplementedError` between `TODO BEGIN` and `TODO END` with your solution. Keep the function name and parameters unchanged, and return the value required by the problem.
3. <strong>Run the check and the connected experiment.</strong> Run your answer cell and then its check cell. Code checks work independently of the observation setting. After writing your prediction for the experiment, change the setting to `RUN_OBSERVATIONS = True`, rerun the setting cell, and run that experiment's cell. After changing your code, rerun the answer, the check, and any experiments that use that function.
4. <strong>Interpret the result.</strong> Compare your prediction with the output. Explain separately whether the check bits are correct, what state the data is in after measurement, and whether the original logical information has been restored.

Check cells display the following statuses.

- <strong>Incomplete:</strong> A TODO has not yet been implemented.
- <strong>Failed:</strong> An execution error occurred or the result does not meet the problem's requirements. Read the message and check your calculation and circuit.
- <strong>Passed:</strong> Your code passed the provided checks.

General observations initially wait with `RUN_OBSERVATIONS = False`. Completing all functions does not enable them automatically. Once the setting is True, run only the experiment cells you need; do not run the entire notebook before writing predictions for new questions. The final application uses a separate `RUN_APPLICATION` setting, so enabling general observations does not also run the application. These two settings are not function TODOs.

Write your explanations separately from the automated code checks. Even when a check says “Passed,” make sure you can explain what was verified and under which conditions.

The small measurement experiments use Qiskit circuits with two or four data qubits and one ancilla. For the full patch, the provided tools calculate ideal stabilizer measurements on the nine-data-qubit state. We verify that the small circuits implement the same measurement principle; you are not asked to build a complete 17-qubit hardware circuit with eight additional ancillas.

Probabilities and state overlaps in this lab come from <strong>exact state calculations</strong>. Under the basic error model, a single syndrome is determined, so its probability is 1 up to small numerical errors. Syndromes are not deterministic for every possible state. The small measurement examples also include inputs for which multiple outcomes are possible.

Follow each tool's instructions for qubit numbering and bit order. Local indices in a small circuit serve a different role from the fixed data-qubit numbers of the full patch. A syndrome tuple is ordered by check name, so do not assume it uses the same order as a raw measurement string.

<strong>You may use AI to study for this lab.</strong> Ask for conceptual explanations, check calculations, or investigate error messages. Run AI-generated code yourself. In written answers, use your actual observations to explain in your own words why the measurement, correction, and verification methods are appropriate.

## Set up the environment

This notebook was validated with Python 3.12, Qiskit 2.3.1, and NumPy 2.2.6. For local use, install the accompanying `requirements.txt` and select that environment as your Jupyter kernel.

If the packages are missing, run the following command in a new code cell, restart the runtime, and run the notebook from the top.

```python
%pip install qiskit==2.3.1 numpy==2.2.6
```

Also run the provided-tools cell below. Long code cells may initially appear collapsed. Each problem explains what the tools do, so you do not need to analyze all of their internals before beginning. Small check circuits calculate conditional states from the actual circuit, while full-patch checks apply ideal projectors to the nine data qubits.

In [ ]:
import sys
from importlib.metadata import version
print("Python", sys.version.split()[0])
for package in ("qiskit", "numpy"):
    print(package, version(package))

### Provided tools

These tools prepare and run circuits and display tables. Run this cell once as well. The public checks support learning; write your explanations separately.

In [ ]:
"""Provided tools: small parity-measurement circuits and ideal measurement/recovery experiments for a nine-data-qubit code.

Full-patch measurements are calculated with projectors on the nine data qubits.
This is not a 17-qubit hardware-circuit experiment with eight additional ancillas.
"""
import itertools
import numbers
from html import escape
import numpy as np
from IPython.display import HTML, display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

ATOL = 1e-9
N_DATA = 9
X_SUPPORTS = ((1, 2), (0, 1, 3, 4), (4, 5, 7, 8), (6, 7))
Z_SUPPORTS = ((0, 3), (1, 2, 4, 5), (3, 4, 6, 7), (5, 8))
CHECK_NAMES = ('a0','a1','a2','a3','b0','b1','b2','b3')
CHECK_STATUS = {}
CHECK_LABELS = ('1b · Stabilizer measurement', '2b · Syndrome prediction', '3b · Correction selection', '4a · Logical fidelity')

def show_table(headers, rows):
    """Display strings and numbers as an HTML table without producing image output."""
    def table_text(value):
        if isinstance(value, (float, np.floating)):
            return f"{value:.6f}"
        if isinstance(value, tuple):
            contents = ", ".join(table_text(item) for item in value)
            return "(" + contents + ("," if len(value) == 1 else "") + ")"
        if isinstance(value, list):
            return "[" + ", ".join(table_text(item) for item in value) + "]"
        return str(value)

    rows = list(rows)
    if any(len(row) != len(headers) for row in rows):
        raise ValueError("Check the number of table columns.")
    heading = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(table_text(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))

def _real(value, label):
    if (not isinstance(value, numbers.Real) or isinstance(value, (bool, np.bool_))
            or not np.isfinite(float(value))):
        raise AssertionError(f"{label}: Return a finite real number.")
    return float(value)

def _real_tuple(value, length, label):
    if not isinstance(value, tuple) or len(value) != length:
        raise AssertionError(f"{label}: Return a tuple of length {length}.")
    return np.array([_real(item, label) for item in value], dtype=float)

def _close(actual, expected, message):
    if not np.allclose(actual, expected, atol=ATOL, rtol=0):
        raise AssertionError(message)

def _run_check(name, check):
    try:
        check()
    except NotImplementedError:
        CHECK_STATUS[name] = "Incomplete"
        print(f"{name}: Incomplete — complete the TODO and run the check again.")
        return False
    except Exception as exc:
        CHECK_STATUS[name] = "Failed"
        print(f"{name}: Failed — {type(exc).__name__}: {exc}")
        return False
    CHECK_STATUS[name] = "Passed"
    print(f"{name}: Passed")
    return True

def _support_mask(support):
    return sum(1 << q for q in support)

def single_error_candidates():
    """One no-error case and 27 single-data-qubit Paulis. Does not select an answer for any syndrome."""
    return (("I", None),) + tuple((pauli, q) for q in range(N_DATA) for pauli in ("X", "Y", "Z"))

def _error_masks(error):
    assert isinstance(error, tuple) and len(error) == 2, "Represent an error/correction as a (Pauli, qubit) tuple."
    pauli, q = error
    if pauli == "I":
        assert q is None, "Represent the identity as ('I', None)."
        return 0, 0
    assert pauli in ("X", "Y", "Z"), "Pauli must be one of I, X, Y, or Z."
    assert isinstance(q, (int, np.integer)) and not isinstance(q, bool) and 0 <= q < N_DATA, "The data-qubit index must be from 0 to 8."
    bit = 1 << int(q)
    return (bit if pauli in ("X", "Y") else 0, bit if pauli in ("Z", "Y") else 0)

_CHECK_MASKS = tuple((_support_mask(s),0) for s in X_SUPPORTS) + tuple((0,_support_mask(s)) for s in Z_SUPPORTS)
_INDEX9 = np.arange(1 << N_DATA)
_PARITY9 = np.array([int(i).bit_count() % 2 for i in _INDEX9])

def _pauli_action(state, x_mask, z_mask):
    state = np.asarray(state, dtype=complex)
    assert state.shape == (512,), "A nine-data-qubit statevector must have length 512."
    output = np.empty_like(state)
    phase = (1j ** (x_mask & z_mask).bit_count()) * (1 - 2 * _PARITY9[_INDEX9 & z_mask])
    output[_INDEX9 ^ x_mask] = phase * state
    return output

def apply_errors(state, errors):
    """Apply the Paulis in errors in their listed order. Wrap even a single error in a list."""
    output = np.asarray(state, dtype=complex).copy()
    for error in errors:
        output = _pauli_action(output, *_error_masks(error))
    return output

def logical_state(label):
    """Provided valid code states. Implementing an encoder circuit is not part of this task."""
    zero = np.zeros(512, dtype=complex)
    zero[0] = 1
    for support in X_SUPPORTS:
        zero = zero + _pauli_action(zero, _support_mask(support), 0)
    zero /= np.linalg.norm(zero)
    one = _pauli_action(zero, _support_mask((0, 3, 6)), 0)
    combinations = {"0": (1, 0), "1": (0, 1), "+": (1, 1), "-": (1, -1), "+i": (1, 1j), "-i": (1, -1j)}
    assert label in combinations, "The logical-state label must be one of 0,1,+,-,+i,-i."
    alpha, beta = combinations[label]
    output = alpha * zero + beta * one
    return output / np.linalg.norm(output)

def ideal_syndrome_measurement(state):
    """Provided: calculate every possible (syndrome, probability, post-measurement state) using ideal projectors.

    Apply (I+(-1)^m S)/2 for each outcome m. Under the basic error model,
    one syndrome has probability 1; multiple outcomes for general states are also supported.
    """
    state = np.asarray(state, dtype=complex)
    assert state.shape == (512,) and np.isclose(np.vdot(state, state).real, 1.0, atol=ATOL, rtol=0), "A normalized nine-data-qubit pure state is required."
    branches = [((), state)]
    for x_mask, z_mask in _CHECK_MASKS:
        next_branches = []
        for prefix, vector in branches:
            transformed = _pauli_action(vector, x_mask, z_mask)
            for bit in (0, 1):
                projected = (vector + (-1)**bit * transformed) / 2
                if np.vdot(projected, projected).real > 1e-14:
                    next_branches.append((prefix + (bit,), projected))
        branches = next_branches
    results = []
    for syndrome, vector in branches:
        probability = float(np.vdot(vector, vector).real)
        results.append({"syndrome": syndrome, "probability": probability, "state": vector / np.sqrt(probability)})
    assert np.isclose(sum(row["probability"] for row in results), 1.0, atol=ATOL, rtol=0)
    return results

def deterministic_syndrome(state):
    """Use for inputs with a deterministic syndrome, as in these experiments."""
    outcomes = ideal_syndrome_measurement(state)
    assert len(outcomes) == 1 and np.isclose(outcomes[0]["probability"], 1, atol=ATOL, rtol=0), "This input allows more than one syndrome."
    return outcomes[0]["syndrome"]


def append_z_parity(qc, n_data):
    """Provided Z-parity collection: data 0..n_data-1 → ancilla n_data."""
    for q in range(n_data):
        qc.cx(q, n_data)


def ancilla_instrument(qc, n_data=None):
    """Data operators K0,K1 for initial ancilla 0 and a final Z measurement."""
    if not isinstance(qc, QuantumCircuit):
        raise AssertionError('Return a QuantumCircuit.')
    if n_data is None:
        n_data = qc.num_qubits - 1
    if n_data not in (2, 4) or qc.num_qubits != n_data + 1 or qc.num_clbits != 0:
        raise AssertionError('Use two or four data qubits and one ancilla, with no classical bits.')
    if qc.parameters:
        raise AssertionError('Do not leave unbound Parameters.')
    forbidden = {'measure','reset','initialize','set_statevector','set_density_matrix','if_else','while_loop','for_loop','switch_case'}
    for instruction in qc.data:
        if instruction.operation.name in forbidden:
            raise AssertionError('The provided tools handle state preparation and measurement. Return only the unitary part.')
    matrix = np.asarray(Operator(qc).data)
    if not np.isfinite(matrix).all() or not np.allclose(np.einsum('ji,jk->ik', matrix.conj(), matrix), np.eye(len(matrix)), atol=ATOL, rtol=0):
        raise AssertionError('A unitary circuit is required.')
    size = 1 << n_data
    return matrix[:size, :size], matrix[size:, :size]


def _same_map(actual, expected):
    coefficient = np.vdot(expected, actual) / np.vdot(expected, expected)
    return bool(np.isclose(abs(coefficient), 1, atol=ATOL, rtol=0)
                and np.allclose(actual, coefficient * expected, atol=ATOL, rtol=0))


def measure_small_check(state, circuit_function, basis, n_data):
    state = np.asarray(state, dtype=complex)
    assert state.shape == (1 << n_data,) and np.isclose(np.vdot(state,state).real, 1)
    maps = ancilla_instrument(circuit_function(basis, n_data), n_data)
    branches = []
    for outcome, matrix in enumerate(maps):
        vector = matrix @ state
        probability = float(np.vdot(vector, vector).real)
        branches.append({'outcome': outcome, 'probability': probability,
                         'state': vector / np.sqrt(probability) if probability > 1e-14 else None,
                         'unnormalized_state': vector})
    assert np.isclose(sum(row['probability'] for row in branches), 1)
    return branches


def recovery_trial(logical_label, errors, decoder):
    """Only the experiment tool receives the actual error. The decoder receives only the syndrome."""
    target = logical_state(logical_label)
    damaged = apply_errors(target, errors)
    outcomes = ideal_syndrome_measurement(damaged)
    assert len(outcomes) == 1, 'This experiment applies a specified Pauli error to a code state.'
    row = outcomes[0]
    correction = decoder(row['syndrome'])
    _error_masks(correction)
    corrected = apply_errors(row['state'], [correction])
    return {'target': target, 'damaged': damaged, 'postmeasurement': row['state'],
            'corrected': corrected, 'syndrome_before': row['syndrome'],
            'syndrome_after': deterministic_syndrome(corrected), 'correction': correction}


def check_check_circuit(function):
    def check():
        for basis in ('Z','X'):
            for n_data in (2,4):
                maps = ancilla_instrument(function(basis, n_data), n_data)
                observable = Operator.from_label(basis * n_data).data
                for m, actual in enumerate(maps):
                    target = (np.eye(1 << n_data) + (-1)**m * observable) / 2
                    assert _same_map(actual, target), 'Check both the outcome bit and the post-measurement data state. Check both basis changes and the CX direction.'
    return _run_check(CHECK_LABELS[0], check)


def check_predict_syndrome(function):
    def check():
        fixtures = (
            ((), (0,0,0,0,0,0,0,0)),
            ((('X',4),), (0,0,0,0,0,1,1,0)),
            ((('Z',7),), (0,0,1,1,0,0,0,0)),
            ((('Y',2),), (1,0,0,0,0,1,0,0)),
            ((('X',1),('X',1)), (0,0,0,0,0,0,0,0)),
            ((('X',6),('X',7)), (0,0,0,0,0,0,0,0)),
        )
        for errors, expected in fixtures:
            actual = function(errors)
            assert isinstance(actual, tuple) and len(actual) == 8, 'Return a tuple of length 8.'
            assert all(isinstance(v, numbers.Integral) and not isinstance(v,(bool,np.bool_)) and v in (0,1) for v in actual)
            assert actual == expected, 'Check the support, error type, and parity of the number of anticommutations.'
    return _run_check(CHECK_LABELS[1], check)


def check_decode_syndrome(function):
    def check():
        code_basis = np.column_stack([logical_state('0'), logical_state('1')])
        for error in single_error_candidates():
            errors = [] if error[0] == 'I' else [error]
            observed = deterministic_syndrome(apply_errors(code_basis[:,0], errors))
            correction = function(observed)
            _error_masks(correction)
            assert correction in single_error_candidates(), 'The correction must be the no-error candidate or one of the single-Pauli candidates.'
            actual = np.column_stack([apply_errors(apply_errors(code_basis[:,j], errors), [correction]) for j in range(2)])
            assert _same_map(actual, code_basis), 'Check that the selected correction preserves both logical basis states and their superpositions.'
    return _run_check(CHECK_LABELS[2], check)


def check_logical_fidelity(function):
    def check():
        plus = np.array([1,1],dtype=complex)/np.sqrt(2)
        plus_i = np.array([1,1j],dtype=complex)/np.sqrt(2)
        minus_i = np.array([1,-1j],dtype=complex)/np.sqrt(2)
        fixtures = (
            (plus_i, plus_i, 1.0), (plus_i, minus_i, 0.0),
            (plus, plus_i, 0.5), (plus_i, 1j*plus_i, 1.0),
            (np.array([1,0]), np.array([np.sqrt(3)/2,1j/2]), 0.75),
            (np.array([1,0,0,0]), np.array([0,0,0,1]), 0.0),
        )
        for target, output, expected in fixtures:
            actual = _real(function(target, output), 'fidelity')
            _close(actual, expected, 'Check complex conjugation, the squared absolute value, and global-phase handling.')
    return _run_check(CHECK_LABELS[3], check)

In [ ]:
LAB_RESULTS = {}


def run_demo(label, experiment, requires=(), requires_observations=True):
    LAB_RESULTS.pop(label, None)
    if requires_observations and not RUN_OBSERVATIONS:
        print(f"[Waiting] {label}: write your prediction, set RUN_OBSERVATIONS to True, and run the setting cell and this experiment.")
        return None
    missing = [key for key in requires if CHECK_STATUS.get(key) != "Passed"]
    if missing:
        print(f"[Waiting] {label}: first pass these code checks: {', '.join(missing)}")
        return None
    try:
        result = experiment()
    except NotImplementedError:
        print(f"[Waiting] {label}: complete the preceding TODOs.")
        return None
    except Exception as error:
        print(f"[Experiment error] {label}: {type(error).__name__}: {error}")
        return None
    LAB_RESULTS[label] = result
    return result


def small_state_text(state):
    if state is None:
        return "Outcome does not occur"
    return np.array2string(state, precision=4, suppress_small=True)


print("Provided tools are ready. Syndrome order: a0,a1,a2,a3,b0,b1,b2,b3.")

In [ ]:
# Write your prediction first, then change this to True and rerun this cell.
RUN_OBSERVATIONS = False

## Problem 1. What can we measure without reading the data values?

In this problem, you will build <strong>a circuit that reads one parity bit through an ancilla while retaining the required superposition in the data after measurement</strong>. We will check both whether the ancilla gives the expected result and whether the data is left in the correct state.

First, compare measuring each data value with measuring only the relation between two values. Then describe that measurement in terms of stabilizers and extend it from a Z-type check to an X-type check.

### Reading individual values versus reading a relation

Suppose the two data qubits are in the state

$$
|\psi\rangle=\alpha|00\rangle+\beta|11\rangle,
\qquad |\alpha|^2+|\beta|^2=1
$$

The ket order is $|q_1q_0\rangle$. For example, $|01\rangle$ denotes the component with q1 equal to 0 and q0 equal to 1.

Measuring both data qubits individually in Z gives either 00 or 11, with probabilities $|\alpha|^2$ and $|\beta|^2$, respectively. After measurement, the data is left in the state corresponding to the observed outcome, so the original superposition between the two components is not preserved.

Instead of reading each value, we want to learn only <strong>whether the two values are the same or different</strong>. The XOR of two bits is 0 when they are equal and 1 when they differ. We will call this one bit the <strong>parity</strong>.

Both 00 and 11 have parity 0. Reading only parity therefore does not distinguish these two components. We can use this to measure an error signature while preserving their relative phase.

Let us express this relation as an observable. Z acts on $|0\rangle$ with sign $+1$ and on $|1\rangle$ with sign $-1$. Writing the tensor product of Z on the two positions as $Z_0Z_1$, the products of the signs are as follows.

| Computational-basis component of the data | Product of the two Z signs | Value of $Z_0Z_1$ | Recorded parity bit |
|---|---|---|---:|
| $\lvert00\rangle$ | $(+1)(+1)$ | $+1$ | 0 |
| $\lvert11\rangle$ | $(-1)(-1)$ | $+1$ | 0 |
| $\lvert01\rangle$ | $(+1)(-1)$ | $-1$ | 1 |
| $\lvert10\rangle$ | $(-1)(+1)$ | $-1$ | 1 |

Thus, <strong>measuring $Z_0Z_1$ gives a $+1/-1$ outcome that indicates whether the two bits are equal or different</strong>. We record these outcomes as bits 0 and 1, respectively.

### Collecting parity in an ancilla

Instead of measuring the data directly, let us transfer only its parity to an ancilla.

Prepare ancilla $a$ in $|0\rangle$. Apply a CX with q0 as control and $a$ as target, followed by a CX with q1 as control and the same $a$ as target.

If the computational-basis data values are $d_0,d_1$, the ancilla flips whenever the corresponding value is 1. Starting from 0, its final value is

$$
0\oplus d_0\oplus d_1=d_0\oplus d_1
$$

Equal data values cause either two flips or no flips, leaving 0. Different values cause one flip, leaving 1.

Now consider the action on a superposition. The expressions below use <strong>the order $|a\rangle\otimes|q_1q_0\rangle$, with the ancilla written first</strong>.

$$
\begin{aligned}
|0\rangle_a
\bigl(\alpha|00\rangle+\beta|11\rangle\bigr)
&\xrightarrow{\mathrm{CX}_{q_0\to a}}
\alpha|0\rangle_a|00\rangle
+\beta|1\rangle_a|11\rangle\\
&\xrightarrow{\mathrm{CX}_{q_1\to a}}
|0\rangle_a
\bigl(\alpha|00\rangle+\beta|11\rangle\bigr).
\end{aligned}
$$

After the first CX, the ancilla values differ between the two components. After the second CX, however, the ancilla is 0 in both. Measuring it does not tell us whether the data component is 00 or 11, and <strong>the data amplitudes and relative phase remain unchanged</strong>.

Now suppose an X error occurs on q0. The data state becomes

$$
\alpha|01\rangle+\beta|10\rangle
$$

Both components have odd parity, so the same circuit produces

$$
|0\rangle_a
\bigl(\alpha|01\rangle+\beta|10\rangle\bigr)
\longmapsto
|1\rangle_a
\bigl(\alpha|01\rangle+\beta|10\rangle\bigr)
$$

The ancilla outcome is now 1, but the two components are still not distinguished.

This check tells us that the parity changed. It does not identify whether the X error occurred on q0 or q1. The next problem uses several checks to obtain more information about errors.

### From parity checks to stabilizers

For the two components $|00\rangle,|11\rangle$, the observable $Z_0Z_1$ has value $+1$ in both cases. Their superposition therefore also satisfies

$$
Z_0Z_1
\bigl(\alpha|00\rangle+\beta|11\rangle\bigr)
=\alpha|00\rangle+\beta|11\rangle
$$

An error-correcting code defines <strong>check conditions shared by all logical states that it protects</strong>. Writing the check operator as $S$ and an encoded logical state as $|\psi_L\rangle$, the valid-state condition is

$$
S|\psi_L\rangle=|\psi_L\rangle
$$

A Pauli operator that leaves valid code states unchanged in this way is called a stabilizer. Measuring it as an observable gives $+1$ on a valid code state.

With several stabilizers, a valid logical state must satisfy all of their conditions simultaneously. The space of states satisfying those conditions is the <strong>code space</strong>.

These conditions must not single out just one logical state. Logical 0, logical 1, and their superpositions must all have the same check values. Otherwise, reading the checks could reveal which logical information was stored.

### Extending a Z-type check to an X-type check

The same method reads Z parity from four data qubits. Prepare the ancilla in 0 and apply a CX from each data qubit to that ancilla. Its final value is

$$
d_0\oplus d_1\oplus d_2\oplus d_3
$$

This indicates whether the number of 1s among the four bits is even or odd. The provided function `append_z_parity(qc, n_data)` performs this parity-collection step.

An X-type check reads <strong>parity in the X basis</strong>. An X outcome of $+1$ corresponds to $|+\rangle$, and an outcome of $-1$ corresponds to $|-\rangle$. The product of several data-qubit X values is therefore determined by how many components correspond to $|-\rangle$: an even number gives $+1$, and an odd number gives $-1$.

To implement this in a circuit, reuse the basis change from Labs 1 and 2:

$$
H|+\rangle=|0\rangle,\qquad
H|-\rangle=|1\rangle.
$$

Applying H to each data qubit maps the two X-basis states to the two Z-basis states. The Z-parity collection circuit can then record the desired parity in the ancilla.

This time, however, we measure only the ancilla, and <strong>the data is retained for later checks and correction</strong>. After collecting the parity, we must therefore restore the data's original basis. Applying H twice is the identity, so another H on each data qubit undoes the initial basis change.

| Step | Z-type check | X-type check |
|---|---|---|
| Prepare the measurement basis | Use the original basis | Map each data qubit's X basis to the Z basis |
| Collect parity | Use the provided Z-parity CX circuit | Use the same Z-parity CX circuit |
| Data basis | Keep the original basis | Restore the original basis |
| Read the outcome | Measure the ancilla in Z | Measure the ancilla in Z |

If you omit the final basis restoration, the ancilla outcome probabilities may still be correct. The data remains in the transformed basis, however, so <strong>the state passed to the next step can differ from the state left by a correct check</strong>.

### Checking the post-measurement state as well as the outcome

An ideal parity measurement retains only the input component associated with the observed outcome. Define the corresponding operators as

$$
P_0=\frac{I+S}{2},\qquad
P_1=\frac{I-S}{2}
$$

Here $I$ is the identity on all the data qubits.

To see why these expressions work, for $S$, denote a state with eigenvalue $+1$ by $|\psi_+\rangle$ and one with eigenvalue $-1$ by $|\psi_-\rangle$. Then

$$
P_0|\psi_+\rangle
=\frac{|\psi_+\rangle+|\psi_+\rangle}{2}
=|\psi_+\rangle,
$$

$$
P_0|\psi_-\rangle
=\frac{|\psi_-\rangle-|\psi_-\rangle}{2}
=0
$$

Thus, $P_0$ retains the $+1$ component and removes the $-1$ component. Conversely, $P_1$ retains the $-1$ component. These operators are called <strong>projectors</strong>.

Writing the outcome bit as $m$ combines the two cases into

$$
P_m=\frac{I+(-1)^mS}{2}
$$

For input $|\psi\rangle$, the probability of outcome $m$ is

$$
p_m=\|P_m|\psi\rangle\|^2
$$

When that probability is positive, the corresponding data state is

$$
|\psi_m\rangle
=\frac{P_m|\psi\rangle}{\sqrt{p_m}}
$$

A branch with probability 0 does not occur, so its normalized post-measurement state is not defined.

If the input already lies entirely in one parity subspace, measuring that parity preserves the state. If it contains components with different parities, the retained component depends on the outcome. Therefore, <strong>a parity measurement does not preserve every possible input state unchanged</strong>. Your circuit must reproduce both the outcome probabilities and the correctly projected data state for each outcome.

### Exercises

### (a) Comparing individual measurements with a parity measurement

Use the following state of two data qubits:

$$
|\psi\rangle
=\frac{\sqrt3}{2}|00\rangle
+\frac{i}{2}|11\rangle.
$$

The ket order is $|q_1q_0\rangle$, and the statevector basis order is $|00\rangle,|01\rangle,|10\rangle,|11\rangle$.

For individual Z measurements of the two data qubits, list the possible outcomes, their probabilities, and the data state after each outcome. Then express the state obtained without distinguishing the measurement outcomes as the probability-weighted average of the branch density matrices. Explain <strong>whether merely ignoring the measurement record afterward restores the original relative phase</strong>.

For the same input, also calculate what happens when an ancilla measures only $Z_0Z_1$. Compare the outcome and the state left in the data.

| Measurement method | Outcomes and probabilities | Information remaining in the data after measurement |
|---|---|---|
| Measure both data qubits individually in Z | … | … |
| Use an ancilla to measure only ZZ parity | … | … |

Explain how the difference in the information obtained by the two methods relates to preservation of the superposition.

### 1a answer

Calculation, observations, and reasoning:

…

### (b) Implementing both the check bit and the post-measurement data state

Implement `check_circuit(basis, n_data)`.

- `basis` is either `"Z"` or `"X"`.
- `n_data` is either 2 or 4.
- Return a `QuantumCircuit(n_data + 1)` with no classical bits and no measurements.

The qubit roles inside this circuit are as follows.

| `n_data` | Local data-qubit indices | Local ancilla index |
|---:|---|---:|
| 2 | q0, q1 | q2 |
| 4 | q0, q1, q2, q3 | q4 |

The execution tool prepares the data and initializes the last qubit, the ancilla, in $|0\rangle$. Connect the provided `append_z_parity(qc, n_data)` with the necessary basis changes and return <strong>the unitary circuit before measurement</strong>. Do not include state preparation, initialization, or measurement gates inside this function.

The execution tool measures the ancilla in Z after your returned circuit. Outcome 0 must correspond to parity $+1$, and outcome 1 to $-1$. In each branch, the data must equal the post-measurement state obtained from the projector $P_m$ defined above.

The check cell tests more than the ancilla outcome probabilities. It also verifies <strong>that no unwanted operation remains on the data and that the correct post-measurement state is produced</strong>. Equivalent circuits implementing the same measurement are allowed, as is a global phase on the whole state within each outcome branch.

In [ ]:
def check_circuit(basis, n_data):
    qc = QuantumCircuit(n_data + 1)
    # TODO BEGIN — Implement the specified parity check and post-measurement state.
    raise NotImplementedError("check_circuit")
    # TODO END
    return qc

In [ ]:
_ = check_check_circuit(check_circuit)

### 1b answer

Calculation, observations, and reasoning:

…

### (c) Does the same ancilla outcome imply the same check?

For two data qubits, compare these circuits:

- The correct X-type check you implemented.
- A circuit <strong>missing the final H gates</strong> that restore the data's original basis.

The second circuit is provided for this observation. The input is

$$
|+\rangle_{q_1}|+\rangle_{q_0}
$$

Before running it, predict whether the two circuits will give the same ancilla outcomes. After running it, complete the table.

| Circuit | Probability of ancilla outcome 0 | Probability of outcome 1 | Post-measurement data state in each possible branch |
|---|---|---|---|
| Correct X-type check | … | … | … |
| Circuit missing the final H gates | … | … | … |

Explain whether these circuits implement the same check. Distinguish the ancilla outcome from the state left in the data.

Next, apply the correct ZZ check to

$$
|+\rangle_{q_1}|0\rangle_{q_0}
$$

List the outcome probabilities and the post-measurement data state for each outcome. Use the parity components in this input to explain why you cannot say that the state always remains unchanged by measurement.

This two-data-qubit experiment illustrates the principle of parity measurement. It does not mean that this check alone can distinguish every error location or correct an arbitrary single-qubit error. In the next problem, we will use several checks based on the same principle on a surface-code patch.

### 1c answer

Prediction before running:

…

Calculation, observations, and reasoning:

…

### Experiment 1c — Check the result using your function

Write your prediction, change the general observation setting to True, and rerun the setting cell. This experiment runs once the required code checks have also passed. After reviewing the result, add your observations and explanation to your answer.

In [ ]:
def experiment_1c():
    def without_return_h(basis, n_data):
        qc = QuantumCircuit(n_data + 1)
        qc.h(range(n_data))
        append_z_parity(qc, n_data)
        return qc

    inputs = [
        ("Correct X check: ++ input", np.ones(4)/2, check_circuit, "X"),
        ("Final H gates omitted: ++ input", np.ones(4)/2, without_return_h, "X"),
        ("Correct ZZ check: +0 input", np.array([1,0,1,0])/np.sqrt(2), check_circuit, "Z"),
    ]
    results, rows = [], []
    for label, state, circuit_function, basis in inputs:
        branches = measure_small_check(state, circuit_function, basis, 2)
        results.append({"label": label, "branches": branches})
        for branch in branches:
            rows.append([label, branch["outcome"], branch["probability"], small_state_text(branch["state"])])
    show_table(["Experiment", "Ancilla outcome", "Probability", "Post-measurement data vector (00,01,10,11)"], rows)
    return results

_ = run_demo("1c", experiment_1c, ("1b · Stabilizer measurement",))

## Problem 2. Which check outcomes does an error change?

In the previous problem, you read parity through an ancilla and checked the data state after measurement. Now you will <strong>use several checks together and calculate which check outcomes an error changes</strong>.

First, examine the surface-code patch and its check table. Then predict syndromes from the commutation relations between checks and errors, and compare those predictions with ideal measurements of an actual code state.

### Why we need several checks

A single ZZ check could not distinguish which of two data qubits had an X error, because an X on either one can change their parity.

To obtain more information about an error, we must <strong>check different combinations of data qubits</strong>. Looking at which outcomes change across several checks can narrow down the error candidates more than a single check can.

We also need more than one type of check to address phase errors. A Z error does not change computational-basis bit values, so a ZZ check alone cannot reveal it. This code therefore uses both Z-type and X-type checks. We will use operator relations below to determine which errors each type responds to.

### Reading the check table for nine data qubits

We use a rotated surface-code patch with nine data qubits arranged as follows.

|  |  |  |
|---|---|---|
| q0 | q1 | q2 |
| q3 | q4 | q5 |
| q6 | q7 | q8 |

This table specifies the data positions and their numbers. The operator table below specifies which data qubits are checked together.

| X-type check | Check operator | Z-type check | Check operator |
|---|---|---|---|
| $A_0$ | $X_1X_2$ | $B_0$ | $Z_0Z_3$ |
| $A_1$ | $X_0X_1X_3X_4$ | $B_1$ | $Z_1Z_2Z_4Z_5$ |
| $A_2$ | $X_4X_5X_7X_8$ | $B_2$ | $Z_3Z_4Z_6Z_7$ |
| $A_3$ | $X_6X_7$ | $B_3$ | $Z_5Z_8$ |

For example, $A_0=X_1X_2$ reads the X parity of q1 and q2. The check $A_1=X_0X_1X_3X_4$ reads the X parity of q0, q1, q3, and q4. The identity acts on data qubits not listed in the operator.

<strong>Writing X on several data qubits does not mean measuring each one individually in X and reading all the individual outcomes.</strong> As in the previous problem, use an ancilla to read one parity bit corresponding to the product of the observable values, while retaining the data for later checks and correction.

The list of data positions on which a check acts is its <strong>support</strong>. The lists in the table are provided in the code as `X_SUPPORTS` and `Z_SUPPORTS`, ordered by check name.

The small circuits in the previous problem used local indices. For example, q0, q1, q2, and q3 in a four-data-qubit circuit label positions within that circuit. For the patch check $A_1$, those four local positions correspond to patch qubits q0, q1, q3, and q4. <strong>Distinguish local circuit indices from the fixed data-qubit numbers of the patch.</strong>

Valid code states have value $+1$ for all eight checks. The code space satisfying these conditions stores one logical qubit.

This code has <strong>distance 3</strong>. A Pauli operation that changes the logical information without being detected by the checks must act on at least three data qubits. With ideal preparation, measurement, and correction, the code can correct any single-data-qubit Pauli error, but it does not guarantee correction of arbitrary two-data-qubit errors. We will first learn to read check outcomes for single errors.

### What changes when we reverse the order of a check and an error?

To predict a check outcome, we need the relation between the error and the check operator.

Two operators $S,E$ <strong>commute</strong> if

$$
SE=ES
$$

They <strong>anticommute</strong> if

$$
SE=-ES
$$

Reversing the order of two anticommuting operators changes the sign.

X and Z acting on the same qubit anticommute:

$$
ZX=-XZ.
$$

Pauli operators of the same type commute. Operators acting on different qubits also commute. For example, when $i\ne j$,

$$
X_iZ_j=Z_jX_i
$$

Let us connect this relation to measurement outcomes. Suppose a valid code state satisfies

$$
S|\psi_L\rangle=|\psi_L\rangle
$$

and an error $E$ acts on it. The state after the error is $E|\psi_L\rangle$.

If $S$ and $E$ commute, then

$$
SE|\psi_L\rangle
=ES|\psi_L\rangle
=E|\psi_L\rangle
$$

The state after the error therefore still has eigenvalue $+1$ for this check.

If instead $S$ and $E$ anticommute, then

$$
SE|\psi_L\rangle
=-ES|\psi_L\rangle
=-E|\psi_L\rangle
$$

The state after the error now has, for check $S$, eigenvalue $-1$.

The sign here is <strong>the eigenvalue obtained by measuring the check</strong>. It does not mean that we distinguish an error state by attaching a global phase to it. In terms of the projectors from the previous problem, this state is removed by $P_0=(I+S)/2$ and retained by $P_1=(I-S)/2$.

Thus, <strong>knowing which checks anticommute with an error lets us predict which outcomes change from $+1$ to $-1$</strong>.

This calculation did not require the logical amplitudes $\alpha,\beta$. Under our current conditions—a specified Pauli error applied to a valid code state—we can calculate changes in check signs without learning the stored logical information.

### Why X-type and Z-type checks can be used together

X and Z on the same position anticommute, but for checks acting on several positions, we must count all the positions they share.

For example, consider

$$
S_X=X_iX_j,\qquad S_Z=Z_iZ_j
$$

Swapping X and Z on qubit q$i$ contributes one minus sign, and swapping them on q$j$ contributes another. Therefore,

$$
S_XS_Z=(-1)^2S_ZS_X=S_ZS_X
$$

The operators anticommute at each shared position, but the full check operators commute.

In our table, each X-type check and Z-type check overlap on either zero or two positions, so they commute. Ideal measurements can therefore verify these conditions together without one check flipping a previously obtained value of another.

Distinguish the following two relations.

- <strong>Between checks:</strong> This determines whether several valid-state conditions can be measured together.
- <strong>Between a check and an error:</strong> This determines which check outcomes the error changes.

### Different error types affect different checks

An X-type check commutes with an X error on a position in its support, but anticommutes with a Z error there. A Z-type check does the reverse: it anticommutes with X and commutes with Z.

Up to a global phase, Y has the same action as applying both X and Z. Its Z component changes the sign of an X-type check, and its X component changes the sign of a Z-type check.

The effects are summarized below.

| Error on a data qubit in the check's support | X-type check | Z-type check |
|---|---|---|
| X error | Sign unchanged | Sign flipped |
| Z error | Sign flipped | Sign unchanged |
| Y error | Sign flipped | Sign flipped |

This table applies <strong>when the error location belongs to the check's support</strong>. An error on a different data qubit commutes with the check and leaves its sign unchanged.

For several errors, count the sign flips for each check. An odd number flips its sign; an even number restores the original sign. For example, X errors at two different positions in one Z-type check can produce two sign flips that cancel.

Do not set a check bit to 1 merely because the check overlaps an error location. You must consider <strong>the check's support, the Pauli error type, and the parity of the number of anticommutations</strong>.

### Recording check signs as syndrome bits

Record a check outcome of $+1$ as 0, and an outcome of $-1$ as 1. Denote the four X-check bits by $a_0,a_1,a_2,a_3$ and the four Z-check bits by $b_0,b_1,b_2,b_3$.

The syndrome order is always

$$
s=(a_0,a_1,a_2,a_3,b_0,b_1,b_2,b_3)
$$

This tuple is <strong>a record ordered by check name</strong>. It does not directly copy the order of a raw Qiskit bitstring; the provided tools arrange the outcomes in this order. Here $a_i,b_i$ record stabilizer measurements, rather than the game outputs used in the previous lab.

### Worked prediction: an X error on the central data qubit

Suppose the central qubit q4 experiences an $X_4$ error.

The X-type checks commute with $X_4$. Even the X-type checks containing q4 meet the same Pauli type there, so their signs do not change. Thus, $a_0,a_1,a_2,a_3$ are all 0.

For the Z-type checks, find the ones containing q4:

$$
B_1=Z_1Z_2Z_4Z_5,\qquad
B_2=Z_3Z_4Z_6Z_7
$$

Both contain Z on q4, while the error is X on the same position, so both signs flip. The checks $B_0,B_3$ do not contain q4, and their signs remain unchanged.

Therefore,

$$
s(X_4)=(0,0,0,0,0,1,1,0)
$$

This record is not a direct measurement of the value of q4. It records <strong>which check conditions changed</strong>. The same reasoning applies to other positions and Pauli error types.

### Comparing a prediction with a state-based measurement calculation

Your syndrome predictor uses the relation between the check table and an error. Independently, the provided measurement tool applies

$$
P_m=\frac{I+(-1)^mS}{2}
$$

to the actual nine-data-qubit state after the error, calculating each outcome's probability and post-measurement state. It performs the eight checks in sequence to obtain the syndrome.

The previous problem verified that your small measurement circuits implement these projectors. We now use the same principle to calculate full-patch outcomes. <strong>The measurement tool does not simply use your predicted syndrome as its result</strong>, so comparing the two provides a check on your syndrome calculation.

The provided tools handle logical-state preparation and the large statevector. Focus on reading the check table and translating the relations between errors and checks into syndrome bits.

### Exercises

### (a) Finding shared positions and error signatures in the check table

First, identify the data positions on which both $A_1$ and $B_1$ act. Count the sign flips at those positions and explain why the two full checks commute.

Next, predict the syndrome when the error on q4 is Z or Y. List the checks whose signs flip, then express the result as an 8-bit tuple in the specified order.

| Error | Checks whose signs flip | Predicted syndrome |
|---|---|---|
| $Z_4$ | … | … |
| $Y_4$ | … | … |

For each row, explain the relation between the error type and the check type. Use commutation and anticommutation; do not assume that a check responds to an error of the same name.

### 2a answer

Calculation, observations, and reasoning:

…

### (b) Writing a syndrome predictor

Implement `predict_syndrome(errors)`.

`errors` is a list or tuple of `(Pauli, data_qubit_index)` pairs. The Pauli is one of `"X"`, `"Y"`, or `"Z"`, and the data-qubit index is from 0 to 8.

For example,

```python
[("X", 4)]
```

means an X error on q4. An empty list or tuple means no error. Multiple errors may be supplied, and the same location may appear more than once.

Account for the number of anticommutations with each check and return <strong>a tuple of length 8 containing integer 0s and 1s</strong>. The return order is always

```python
(a0, a1, a2, a3, b0, b1, b2, b3)
```

Use the provided `X_SUPPORTS` and `Z_SUPPORTS`. Your calculation must work for different error locations and types, rather than returning only the central-X example. When several errors affect one check, account for whether the number of sign flips is even or odd.

This function <strong>analyzes a known error or investigates correction candidates</strong>. It does not mean that the decoder in the next problem receives the actual error. Also distinguish the ability to predict a multi-error syndrome from the ability to correct all such errors.

In [ ]:
def predict_syndrome(errors):
    # TODO BEGIN — Use the check table and error list to return an 8-bit tuple.
    raise NotImplementedError("predict_syndrome")
    # TODO END

In [ ]:
_ = check_predict_syndrome(predict_syndrome)

### 2b answer

Calculation, observations, and reasoning:

…

### (c) Comparing predicted and measured check outcomes

Use the provided tools to apply each error below to a code state and calculate the ideal stabilizer measurement outcomes. Use the logical input

$$
|+i_L\rangle
=\frac{|0_L\rangle+i|1_L\rangle}{\sqrt2}
$$

which includes a relative phase.

First predict each syndrome, then compare your function's output with the measurement tool's result.

| Actual error | Syndrome predicted by your function | Syndrome from the provided measurement tool |
|---|---|---|
| $Z_4$ | … | … |
| $Y_4$ | … | … |
| $X_1$ | … | … |
| $X_2$ | … | … |

Check whether the two calculations agree. For any disagreement, review the checks containing that location, anticommutation for the Pauli type, and the order of the returned bits.

Then find any rows where different errors produce the same syndrome. Use your results to explain <strong>whether a syndrome always uniquely identifies the actual error's location and type</strong>.

The next problem asks whether we can choose a correction that restores the original logical information even when several candidates share a syndrome.

### 2c answer

Prediction before running:

…

Calculation, observations, and reasoning:

…

### Experiment 2c — Check the result using your function

Write your prediction, change the general observation setting to True, and rerun the setting cell. This experiment runs once the required code checks have also passed. After reviewing the result, add your observations and explanation to your answer.

In [ ]:
def experiment_2c():
    target = logical_state("+i")
    rows, results = [], []
    for error in (("Z",4), ("Y",4), ("X",1), ("X",2)):
        prediction = predict_syndrome([error])
        observed = deterministic_syndrome(apply_errors(target, [error]))
        rows.append([error, prediction, observed, prediction == observed])
        results.append({"error": error, "prediction": prediction, "observed": observed})
    show_table(["Actual error", "Your prediction", "Independent state-based measurement", "Match"], rows)
    return results

_ = run_demo("2c", experiment_2c, ("2b · Syndrome prediction",))

## Problem 3. What should we correct when several errors share a syndrome?

The previous problem showed that different errors can produce the same syndrome. Now you will implement <strong>a decoder that receives only the syndrome and selects a correction</strong>.

First, consider whether failing to identify the actual error uniquely necessarily prevents correction. Then use the criterion for successful recovery to choose a candidate and verify that logical information is restored for no error and every single-data-qubit Pauli error.

### How should we judge whether a correction succeeds?

Write the valid logical state as $|\psi_L\rangle$ and the actual error as $E$. After the error, the data state is

$$
E|\psi_L\rangle
$$

As discussed previously, a specified Pauli error applied to a valid code state determines each stabilizer's value. An ideal syndrome measurement reads those values and leaves the data state unchanged, because it already has definite values for the checks.

Applying correction $R$ then produces

$$
RE|\psi_L\rangle
$$

The operator $E$ on the right acts first, and $R$ on the left acts afterward. To evaluate the correction, examine what <strong>the combined error-and-correction operation $RE$</strong> does to the logical state.

The simplest case is to apply the same Pauli as the error at the same location. For example, if the error is $X_i$ and the correction is also $X_i$,

$$
X_iX_i=I
$$

so the original state is restored. Applying Y twice or Z twice also gives the identity.

This is not the only possible correction. <strong>If the operation remaining after correction leaves every valid logical state unchanged, a correction different from the actual error can also succeed.</strong>

### A remaining stabilizer leaves the logical state unchanged

For each stabilizer $S$, a valid code state satisfies

$$
S|\psi_L\rangle=|\psi_L\rangle
$$

If the combined error and correction is a stabilizer, then

$$
RE|\psi_L\rangle
=S|\psi_L\rangle
=|\psi_L\rangle
$$

Products of stabilizers have the same property. For example, if $S_1,S_2$ each leave valid logical states unchanged,

$$
S_1S_2|\psi_L\rangle
=S_1|\psi_L\rangle
=|\psi_L\rangle
$$

Thus, logical information is restored whenever $RE$ is a stabilizer or a product of stabilizers. An overall global phase also represents the same physical state.

The important point is that <strong>this property holds for every logical state in the code space</strong>, not merely that one particular logical input happens to remain unchanged.

### Worked example: correcting an X1 error with X2

In the previous problem, both $X_1$ and $X_2$ changed only the sign of $B_1$. Their syndromes are identical:

$$
s(X_1)=s(X_2)
=(0,0,0,0,0,1,0,0).
$$

A decoder receiving only this syndrome cannot distinguish whether the actual error was $X_1$ or $X_2$.

Suppose it chooses $X_2$ as the correction. If the actual error was $X_2$, then

$$
X_2X_2=I
$$

and the original state is restored.

If the actual error was $X_1$, the remaining operation is

$$
X_2X_1=X_1X_2=A_0
$$

X operators on different locations commute, and $A_0=X_1X_2$ is a stabilizer of this code. Therefore, in this case too,

$$
X_2X_1|\psi_L\rangle
=A_0|\psi_L\rangle
=|\psi_L\rangle
$$

| Actual error $E$ | Chosen correction $R$ | Remaining operation $RE$ | Action on a valid logical state |
|---|---|---|---|
| $X_1$ | $X_2$ | $A_0$ | Leaves the state unchanged. |
| $X_2$ | $X_2$ | $I$ | Leaves the state unchanged. |

The same correction $X_2$ succeeds in both cases. Choosing $X_1$ instead would also restore the logical state in both cases.

We therefore treat <strong>corrections that restore the same logical information as equivalent</strong>. Uniquely locating a physical error and restoring the stored logical information are not always the same task.

### When is it safe to choose a candidate with the same syndrome?

Can we choose any operation with the same syndrome as a correction? The answer depends on <strong>which error candidates are allowed</strong>.

Our basic error model allows no error or an X, Y, or Z error on one data qubit. If both the actual error and the correction candidate lie within this set, their combined operation acts on at most two data qubits.

When their syndromes are equal, the correction reverses each check-sign change caused by the error, so every check returns to its expected sign. Now use the fact that this code has distance 3. As explained in the previous problem, <strong>a Pauli operation that changes logical information without being detected by the checks must act on at least three data qubits</strong>.

The combined operation of two candidates in this set therefore cannot both preserve the expected check outcomes and change the logical information. In this code, single-error candidates with the same syndrome differ by a stabilizer, and choosing either one restores the logical information.

This guarantee requires <strong>the actual error also to be at most one data-qubit Pauli error</strong>. Outside that assumption, choosing a single-error candidate with the same syndrome does not by itself establish successful recovery. We will examine that situation in the next problem.

### Finding a correction candidate from a syndrome

There is one no-error case, and each of the nine data qubits has three possible single Pauli errors: X, Y, and Z. The number of candidates is therefore

$$
1+9\times3=28
$$

The provided function `single_error_candidates()` returns them. Each candidate is represented as follows.

| Candidate representation | Meaning |
|---|---|
| `("I", None)` | The identity: apply no correction |
| `("X", 4)` | Apply X to q4 |
| `("Y", 2)` | Apply Y to q2 |
| `("Z", 5)` | Apply Z to q5 |

Investigate the candidates as follows.

1. Select a candidate.
2. Use your `predict_syndrome()` function to calculate the syndrome it would produce.
3. Compare that prediction with the received syndrome.
4. If they match, choose that candidate as the correction.

When several candidates share a syndrome, choose any one of them. Having 28 candidates does not necessarily mean there are 28 distinct syndromes.

<strong>Distinguish predicting the effect of a known actual error from investigating possible candidates.</strong> The decoder does not receive the actual error. It knows the check table and candidate list, so it calculates which syndrome would have occurred if each candidate had been the actual error.

| Role | Available information |
|---|---|
| Experiment tool that injects errors | The initial state and the actual injected error |
| Function that predicts a candidate's syndrome | The candidate being examined and the check table |
| Decoder | The measured syndrome, the code's check table, and the candidate list |
| Tool that verifies recovery | The initial state, selected correction, and corrected state |

This decoder returns the name of a correction operation, and the provided execution tool applies it to the data. The data remains encoded during correction, so this procedure must be distinguished from undoing the encoding circuit.

### Preserving superpositions as well as the logical basis states

For a code storing one logical qubit, an arbitrary logical state can be written as

$$
|\psi_L\rangle
=\alpha|0_L\rangle+\beta|1_L\rangle
$$

Suppose a correction leaves <strong>the same global phase</strong> on the two logical basis states:

$$
RE|0_L\rangle=e^{i\phi}|0_L\rangle,
\qquad
RE|1_L\rangle=e^{i\phi}|1_L\rangle
$$

By linearity,

$$
\begin{aligned}
RE|\psi_L\rangle
&=\alpha RE|0_L\rangle+\beta RE|1_L\rangle\\
&=e^{i\phi}
\bigl(\alpha|0_L\rangle+\beta|1_L\rangle\bigr)
\end{aligned}
$$

The superposition is therefore restored to the same physical state as well.

If the two basis states acquire different phases, checking each basis state separately may not reveal a change, even though a superposition's relative phase has changed. The provided code check therefore does more than verify that each logical basis state individually appears unchanged: it also checks <strong>whether one common global phase describes the action on both basis states</strong>.

Distinguish a syndrome returning to zero from logical information being restored in this stronger sense.

### Exercises

### (a) Deciding whether a different correction restores the same logical state

Use your predictor or the check table to calculate the syndromes of $Z_0$ and $Z_3$.

Suppose the actual error is $Z_0$ and the selected correction is $Z_3$. Answer the following questions.

- Do the two errors have the same syndrome?
- What operation remains after applying the correction following the actual error?
- Which stabilizer in the check table does that operation correspond to?

Use this relation to explain whether an arbitrary valid logical state is restored. Also explain whether different names for the actual error and the correction are sufficient to conclude that recovery failed.

### 3a answer

Calculation, observations, and reasoning:

…

### (b) Writing a decoder that uses only the syndrome

Implement `decode_syndrome(syndrome)`.

The input `syndrome` is guaranteed to be <strong>a tuple of eight integers</strong> that can arise from no error or a single-data-qubit Pauli error. As before, the order is

```python
(a0, a1, a2, a3, b0, b1, b2, b3)
```

Examine the candidates provided by `single_error_candidates()` and use your `predict_syndrome()` function to find one whose syndrome matches the received syndrome.

Return a `(Pauli, data_qubit_index)` tuple, for example:

```python
("X", 4)
```

Represent the identity correction as

```python
("I", None)
```

Distinguish the input formats of these functions.

- Each element returned by `single_error_candidates()` is a tuple representing <strong>one candidate</strong>.
- `predict_syndrome()` accepts <strong>a list or tuple of errors</strong>.

For an ordinary single-error candidate, wrap it in a list, as in `[candidate]`. For the no-error candidate `("I", None)`, use an empty error sequence, as in `predict_syndrome(())`.

If several candidates match the syndrome, you may select any of them, including the first or last. The returned correction must be one of the 28 provided candidates. You may raise an exception for a syndrome outside the promised input set.

<strong>Do not pass the actual error or the initially prepared logical state as additional inputs to the function.</strong> Choose a correction using only the received syndrome and the code's candidate information.

In [ ]:
def decode_syndrome(syndrome):
    # TODO BEGIN — Select the no-error or a single-error correction candidate from the syndrome.
    raise NotImplementedError("decode_syndrome")
    # TODO END

In [ ]:
_ = check_decode_syndrome(decode_syndrome)

### 3b answer

Calculation, observations, and reasoning:

…

### (c) Checking recovery for no error and all single errors

Connect your decoder to the provided repeated experiment. For no error and each of the 27 single X, Y, or Z errors, check the following.

- The syndrome obtained from the actual measurement calculation.
- The correction chosen by the decoder.
- The syndrome after correction.
- The logical-information preservation result from the provided check cell.

After reviewing all the results, choose one case in which <strong>the selected correction differs from the actual error</strong> and complete the table.

| Actual error $E$ | Measured syndrome | Chosen correction $R$ | Remaining operation $RE$ | Why the logical state is restored |
|---|---|---|---|---|
| … | … | … | … | … |

Explain which stabilizer accounts for the difference. State why a decoder that chooses another equivalent candidate for the same syndrome can also be correct.

Finally, summarize the assumptions behind the recovery guarantee demonstrated here. Include the allowed actual errors and the assumptions about measurements and correction. <strong>Do not use a zero syndrome alone as evidence that the logical information was restored.</strong> Use both the provided state-preservation check and the operator relations above.

In the next problem, we will examine which correction the same decoder chooses when the actual error falls outside these assumptions. We will calculate the corrected syndrome and the overlap with the original logical state separately to determine what justifies a claim of successful recovery.

### Experiment 3c — Check the result using your function

Write your prediction, change the general observation setting to True, and rerun the setting cell. This experiment runs once the required code checks have also passed. After reviewing the result, add your observations and explanation to your answer.

In [ ]:
def experiment_3c():
    rows, results = [], []
    for error in single_error_candidates():
        errors = [] if error[0] == "I" else [error]
        trial = recovery_trial("+i", errors, decode_syndrome)
        rows.append([error, trial["syndrome_before"], trial["correction"], trial["syndrome_after"]])
        results.append({"error": error, **trial})
    show_table(["Actual error", "Measured syndrome", "Your selected correction", "Syndrome after correction"], rows)
    print("The correction-selection check above verifies state preservation, including one common phase for both logical basis states.")
    return results

_ = run_demo("3c", experiment_3c, ("2b · Syndrome prediction", "3b · Correction selection"))

### 3c answer

Calculation, observations, and reasoning:

…

## Problem 4. Does a zero syndrome mean that recovery succeeded?

In the previous problem, you built a correction rule under the assumption of at most one data-qubit Pauli error. Now you will <strong>directly calculate whether the corrected state contains the same logical information as the original state</strong>.

First, consider why a state that passes all checks may still differ from the state originally stored. Then introduce errors on two data qubits and compare the results of using your existing decoder on different logical inputs.

### Being in the code space is different from being in the original logical state

The code space contains not only $|0_L\rangle$ but also $|1_L\rangle$ and their superpositions. All these states have value $+1$ for each of the eight stabilizers.

A syndrome consisting entirely of zeros therefore tells us that <strong>the data lies in the code space and satisfies the expected check conditions</strong>. It does not yet tell us whether the state within that space is the original logical state.

For example, suppose we initially stored $|0_L\rangle$ but the output became $|1_L\rangle$. Both states satisfy the expected check conditions, so the syndrome alone does not reveal the difference. The stored information has nevertheless changed.

An operation that changes a logical state within the code space is called a <strong>logical operation</strong>. Logical X exchanges the two logical basis states:

$$
\overline X|0_L\rangle=|1_L\rangle,
\qquad
\overline X|1_L\rangle=|0_L\rangle.
$$

For this patch, we can use

$$
\overline X=X_0X_3X_6
$$

This operation commutes with every stabilizer, maintaining a zero syndrome while exchanging the logical basis states.

Distinguish this from the stabilizer action in the previous problem. A stabilizer leaves every valid logical state unchanged. Logical X preserves the code space but changes the information within it.

A logical operation can also change the relative phase between the logical basis states. Logical Z is defined by

$$
\overline Z|0_L\rangle=|0_L\rangle,
\qquad
\overline Z|1_L\rangle=-|1_L\rangle
$$

Preserving logical information therefore requires preserving both the logical basis values and the relative phase between them.

### A residual error can be invisible on some inputs

A remaining logical X changes $|0_L\rangle$ to $|1_L\rangle$, but it does not change every input in the same way.

Consider the logical state $|+_L\rangle$:

$$
|+_L\rangle
=\frac{|0_L\rangle+|1_L\rangle}{\sqrt2}.
$$

Applying logical X gives

$$
\begin{aligned}
\overline X|+_L\rangle
&=\frac{\overline X|0_L\rangle+\overline X|1_L\rangle}{\sqrt2}\\
&=\frac{|1_L\rangle+|0_L\rangle}{\sqrt2}\\
&=|+_L\rangle
\end{aligned}
$$

The two terms are merely exchanged, so the state stays the same.

Testing only $|+_L\rangle$ could therefore miss the effect of a remaining logical X. The result means <strong>that this input was preserved</strong>, not that the correction process preserves every logical input.

This is why the previous problem compared the action on both logical basis states using one common global phase. To protect arbitrary logical information, it is not enough to see no change in one particular state.

### Calculating fidelity between the target and output states

We will now quantify how closely the corrected state matches the original state.

In these experiments, a <strong>specified Pauli error</strong> is applied to a normalized pure logical input. Each experiment has a definite syndrome, and the output after ideal measurement and correction can also be represented by a pure-state vector.

Let the target logical state be $|\psi_L\rangle$ and the corrected state be $|\phi\rangle$. Their fidelity is

$$
F=|\langle\psi_L|\phi\rangle|^2
$$

Writing their vector components as $\psi_j,\phi_j$, the inner product is

$$
\langle\psi_L|\phi\rangle
=\sum_j\psi_j^*\phi_j
$$

The asterisk denotes complex conjugation. <strong>Conjugate each component of the target vector, multiply it by the corresponding component of the output, sum the products, and square the absolute value.</strong>

For normalized states, fidelity lies between 0 and 1.

| Fidelity | Meaning |
|---|---|
| $F=1$ | The states are identical up to a global phase. |
| $F=0$ | The states are orthogonal. |
| $0<F<1$ | The states overlap but are not identical. |

Because states with relative phases must also be checked, do not omit complex conjugation. Also remember that the absolute value of an inner product and its square are different quantities.

### Worked check: an output differing only by a global phase

Suppose the corrected state is the original state multiplied by $i$:

$$
|\phi\rangle=i|\psi_L\rangle
$$

Since the state is normalized,

$$
\langle\psi_L|\phi\rangle
=i\langle\psi_L|\psi_L\rangle=i
$$

and

$$
F=|i|^2=1
$$

The vector components may look different under a direct comparison, but the physical state is the same. Such an output must not be classified as a failed recovery.

This fidelity formula applies <strong>when both inputs are pure-state vectors</strong>. Do not apply the same vector expression directly to a general mixed state obtained by combining measurement outcomes without distinguishing them.

### Separating the information available to the verifier and the decoder

To test recovery in this lab, we know which logical state was prepared. The provided tool returns that target state and the corrected state, and your function calculates their fidelity.

It is <strong>the tool preparing and verifying the experiment</strong> that knows the target state. This information is not passed to the decoder. The decoder still chooses a correction using only the syndrome.

The provided tool

```python
recovery_trial(logical_label, errors, decode_syndrome)
```

performs these steps:

1. Prepare the logical state specified by `logical_label`.
2. Apply the errors specified by `errors`.
3. Obtain a syndrome through ideal stabilizer measurements.
4. Pass only that syndrome to `decode_syndrome()`.
5. Apply the returned correction and collect the results.

This problem uses the following returned fields.

| Field | Meaning |
|---|---|
| `target` | The initially prepared logical-state vector |
| `corrected` | The statevector after the selected correction |
| `syndrome_before` | The measured syndrome before correction |
| `correction` | The correction actually selected by the decoder |
| `syndrome_after` | The syndrome after correction |

The full data state has 512 components in $|q_8\cdots q_0\rangle$ order. You do not need to expand this vector manually or implement an encoder. Apply your fidelity function to the two supplied vectors.

### What does the same decoder do when there are two errors?

Now apply X errors to q0 and q3:

$$
E=X_0X_3.
$$

Code-state preparation, measurements, and corrections remain ideal. The check table and decoder are unchanged. The change is that <strong>the actual data error now affects two qubits instead of one</strong>.

The decoder does not receive the number of actual errors. If a single-error candidate matches the received syndrome, it chooses that candidate according to the rule you already implemented.

A match with a single-error candidate's syndrome is not evidence that a single error actually occurred: different errors can produce the same syndrome.

We must therefore check each of the following separately.

- Which syndrome was obtained?
- Which correction did the decoder choose?
- Did the corrected check outcomes return to their expected values?
- Is the corrected state the same as the original logical state?

Record the first three observations separately from the final one.

### Exercises

### (a) Implementing a logical-state recovery check

Implement `logical_fidelity(target_state, corrected_state)`.

Both inputs are normalized complex pure-state vectors of the same length. Return one real number representing their fidelity.

The same calculation must apply to states differing only by global phase, orthogonal states, and states with relative phases. Do not return values that work only for specific inputs or only in dimension 512.

After completing the function, compare the following two outputs using target $|0_L\rangle$.

| Output state | Stabilizer check outcomes | Fidelity with the target |
|---|---|---|
| $\lvert0_L\rangle$ | … | … |
| $\lvert1_L\rangle$ | … | … |

Explain what each type of check verifies. Use the relation between the code space and logical information to explain why states with all expected stabilizer values can nevertheless have different fidelities.

In [ ]:
def logical_fidelity(target_state, corrected_state):
    # TODO BEGIN — Return the fidelity of the two normalized pure states.
    raise NotImplementedError("logical_fidelity")
    # TODO END

In [ ]:
_ = check_logical_fidelity(logical_fidelity)

### Experiment 4a — Check the result using your function

Write your prediction, change the general observation setting to True, and rerun the setting cell. This experiment runs once the required code checks have also passed. After reviewing the result, add your observations and explanation to your answer.

In [ ]:
def experiment_4a():
    target = logical_state("0")
    results, rows = [], []
    for label in ("0", "1"):
        output = logical_state(label)
        syndrome = deterministic_syndrome(output)
        score = logical_fidelity(target, output)
        rows.append([label, syndrome, score])
        results.append({"output_label": label, "syndrome": syndrome, "fidelity": score})
    show_table(["Output logical state", "Syndrome", "F with target logical 0"], rows)
    return results

_ = run_demo("4a", experiment_4a, ("4a · Logical fidelity",))

### 4a answer

Calculation, observations, and reasoning:

…

### (b) Applying the existing decoder to two errors

Set the error list to

```python
errors = [("X", 0), ("X", 3)]
```

Run separate experiments for logical inputs `"0"` and `"+"`. Do not modify `decode_syndrome()` from Problem 3.

Apply your `logical_fidelity()` to the returned `target` and `corrected` vectors and complete the table.

| Logical input | Syndrome before correction | Selected correction | Syndrome after correction | $F$ with the original input |
|---|---|---|---|---|
| $\lvert0_L\rangle$ | … | … | … | … |
| $\lvert+_L\rangle$ | … | … | … | … |

Write your predictions before running the experiment, then compare them with the results.

Next, consider the actual error $E=X_0X_3$ followed by the selected correction $R$, and write the remaining operation $RE$. Relate that operation to a logical operation or stabilizer and explain the results for the two logical inputs.

Also consider what would happen if your decoder chose another equivalent single-error candidate for the same syndrome. Distinguish a change in the correction's name from a change in the residual logical action.

### 4b answer

Prediction before running:

…

Calculation, observations, and reasoning:

…

### Experiment 4b — Check the result using your function

Write your prediction, change the general observation setting to True, and rerun the setting cell. This experiment runs once the required code checks have also passed. After reviewing the result, add your observations and explanation to your answer.

In [ ]:
def experiment_4b():
    errors = [("X",0), ("X",3)]
    results, rows = [], []
    for label in ("0", "+"):
        trial = recovery_trial(label, errors, decode_syndrome)
        score = logical_fidelity(trial["target"], trial["corrected"])
        results.append({"logical_label": label, "fidelity": score, **trial})
        rows.append([label, trial["syndrome_before"], trial["correction"], trial["syndrome_after"], score])
    show_table(["Logical input", "Syndrome before correction", "Selected correction", "Syndrome after correction", "F"], rows)
    return results

_ = run_demo("4b", experiment_4b, ("2b · Syndrome prediction", "3b · Correction selection", "4a · Logical fidelity"))

### (c) Reviewing a claim of successful recovery

Review the following report in light of your results in part (b).

> The corrected syndrome returned to all zeros. The fidelity was also 1 for the logical + input, so we concluded that this correction rule restored the original logical information.

Separate what the report actually established from what cannot be concluded from that evidence alone.

- What does a syndrome of all zeros establish?
- What does fidelity 1 for the logical + input establish?
- Considering the logical 0 result as well, can we claim that arbitrary logical information was restored?
- How does the difference between the two inputs relate to the operation remaining after correction?

Then explain whether this result contradicts the single-error correction guarantee in Problem 3. State the earlier assumptions and the condition that changed in this experiment precisely.

<strong>A lack of a guarantee for arbitrary two-qubit errors is different from a claim that every two-qubit error must fail to be corrected.</strong> Distinguish the result for the specific errors tested here from the guarantee offered by the code in general.

### 4c answer

Calculation, observations, and reasoning:

…

## Final application — What changes if the two errors are Z-type errors?

Keep the same code, ideal measurements, and decoder, but change only the error list to

```python
errors = [("Z", 0), ("Z", 1)]
```

Again, the actual errors act on two data qubits. Do not create a new decoder or modify the existing correction rule.

Before running the experiment, predict the syndrome and a matching single-error correction candidate. Then test logical inputs `"0"` and `"+"` separately and use your fidelity function to check the results.

| Logical input | Predicted correction candidate | Actually selected correction | Syndrome after correction | $F$ with the original input |
|---|---|---|---|---|
| $\lvert0_L\rangle$ | … | … | … | … |
| $\lvert+_L\rangle$ | … | … | … | … |

Compare this with the earlier two-X-error case and explain the following.

- Do the check outcomes return to their expected values after correction?
- Which logical input reveals the remaining error, and which one hides it?
- How does the combined actual error and selected correction act on the logical state?

After finding the remaining operation, check <strong>how it acts on each logical basis state</strong>. Obtain the basis states with `logical_state("0")` and `logical_state("1")`, then use `apply_errors(state, error_list)` to apply the actual errors followed by the selected correction. Comparing the inner products of each input and output also lets you determine the overall phase acquired by each state.

Even if both basis states return to their original directions, distinguish whether the two outputs acquire <strong>the same phase</strong> or <strong>different phases that introduce a relative phase</strong>. A common phase means every superposition is also restored up to that global phase. Different phases can change a superposition. Also distinguish cases where an output is mapped to a different basis state. Relate the two fidelities above to this action on the basis states.

No new function or circuit is needed. Reuse your tools and briefly explain <strong>the difference between preserving one particular input and restoring arbitrary logical information</strong>.

## Final application answer

Prediction before running:

…

Calculation, observations, and reasoning:

…

### Check the application result

First write your prediction and reasoning. Then change `RUN_APPLICATION` to `True` in the cell below and run it. The result is calculated using the functions you implemented. This setting is not a new function TODO.

In [ ]:
RUN_APPLICATION = False


def application_experiment():
    errors = [("Z",0), ("Z",1)]
    results, rows = [], []
    for label in ("0", "+"):
        prediction = predict_syndrome(errors)
        trial = recovery_trial(label, errors, decode_syndrome)
        score = logical_fidelity(trial["target"], trial["corrected"])
        rows.append([label, prediction, trial["syndrome_before"], trial["correction"], trial["syndrome_after"], score])
        results.append(dict(trial, logical_label=label, fidelity=score, prediction=prediction))
    show_table(["Logical input", "Your prediction", "Measured syndrome", "Selected correction", "Syndrome after correction", "F"], rows)
    return results


LAB_RESULTS.pop("application", None)
if RUN_APPLICATION:
    _ = run_demo("application", application_experiment,
                 ("2b · Syndrome prediction", "3b · Correction selection", "4a · Logical fidelity"), requires_observations=False)
else:
    print("Write your prediction before running. To check it, set RUN_APPLICATION to True and run this cell.")

## Lab 3 wrap-up

Check the status of all four code checks and review your written answers. Finally, summarize the conditions for judging successful error correction in three sentences. Consider measurement outcomes and post-measurement states, the decoder's available information and error assumptions, and the distinction between restoring a zero syndrome and restoring the logical state.

### Wrap-up answer

Calculation, observations, and reasoning:

…

In [ ]:
expected_checks = ['1b · Stabilizer measurement', '2b · Syndrome prediction', '3b · Correction selection', '4a · Logical fidelity']
show_table(["Code check", "Status"], [[name, CHECK_STATUS.get(name, "Not run")] for name in expected_checks])
if all(CHECK_STATUS.get(name) == "Passed" for name in expected_checks):
    print("All four code checks passed. Review your written explanations and final-application answer separately.")
else:
    print("Review items marked Incomplete, Not run, or Failed. Rerun the updated answer and check cells, then the connected experiments.")